## Installs

In [0]:
%pip install unidecode --quiet

## Imports

In [0]:
from unidecode import unidecode
import unicodedata
import re
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

In [0]:
def get_catalog():
    workspace = spark.conf.get("spark.databricks.workspaceUrl")
    if "2148885194133801" in workspace:
        return "ta_coll"
    elif "2743854327825858" in workspace:
        return "ta_prod"
    else:
        raise Exception(f"Workspace non riconosciuto: {workspace}")

def get_llm_endpoint():
    workspace = spark.conf.get("spark.databricks.workspaceUrl")
    if "2148885194133801" in workspace:
        return "https://llm-whatifp-coll.openai.azure.com/openai/v1/"
    elif "2743854327825858" in workspace:
        return "https://llm-whatifp-prod.openai.azure.com/openai/v1/"
    else:
        raise Exception(f"Workspace non riconosciuto: {workspace}")

In [0]:
def _strip_accents(s):
    """à -> a, è -> e (tivu.tv usa diacritici, Auditel li perde)."""
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

# Suffissi editoriali — port da competitor_features._SUFFIXES (in minuscolo)
# + i pattern emersi nel debug del job 11.
_TITLE_SUFFIXES = [
    # Edizioni TG / pagine
    r"\s+edizione\s+straordinaria$", r"\s+ed\s+straordinaria$",
    r"\s+prima\s+pagina$", r"\s+breaking\s+news$",
    r"\s+ultim[ae]?\s+ora(\s+\w+)?$",
    r"\s+ore\s+\d{1,2}(\s+\d{1,2})?$",
    # Stagioni
    r"\s+prima\s+stagione.*$", r"\s+seconda\s+stagione.*$",
    r"\s+terza\s+stagione.*$", r"\s+quarta\s+stagione.*$",
    r"\s+stagione\s+\d+.*$",
    # Cicli stagionali / weekend
    r"\s+il\s+weekend.*$", r"\s+weekend.*$",
    r"\s+estate$", r"\s+cronache\s+d\s+estate$",
    r"\s+sabato$", r"\s+domenica$", r"\s+di\s+piu$",
    # Edizioni per accessibilità (lingua dei segni)
    r"\s+lis$",
    # Speciali / varianti
    r"\s+speciale.*$",
    r"\s+1\^?\s*visione$", r"\s+prima\s+visione$",
    r"\s+inizia\s+la\s+sfida.*$", r"\s+le\s+\d+\s+botole.*$",
    r"\s+prima\s+sfida$", r"\s+il\s+torneo\s+dei\s+campioni$",
    r"\s+il\s+torneo$", r"\s+cosa\s+vi\s+siete\s+persi$",
    # Eventi politici / cronaca
    r"\s+elezioni.*$", r"\s+referendum$", r"\s+si\s+no$", r"\s+si\s+o\s+no$",
    r"\s+il\s+bis\s+di\s+trump.*$", r"\s+il\s+ritorno\s+di\s+trump.*$",
    r"\s+la\s+morte\s+del\s+papa$",
    r"\s+diario\s+del\s+giorno.*$", r"\s+diario\s+della.*$",
    # Closing / segment markers
    r"\s+highlights$", r"\s+aftershow$", r"\s+after\s+show$",
    r"\s+buonanotte$", r"\s+saluti$", r"\s+i\s+saluti$",
    r"\s+rewind$", r"\s+compilation.*$", r"\s+tra\s+poco$",
    # Eventi sportivi numerati (Giro d'Italia 109^ edizione 12^ tap)
    r"\s+\d+\s*edizione.*$", r"\s+\d+\s*tap.*$",
    # Anno trailing (Giro d'Italia 2026 -> giro ditalia)
    r"\s+\d{4}$",
]

# Prefissi editoriali — port da get_family_key
_TITLE_PREFIXES = [
    r"^pres\.?\s*",
    r"^anteprima\s+",
    r"^ant\.?\s*",
    r"^i\s+saluti\s+di\s+",
    r"^la\s+buonanotte\s+di\s+",
]

def normalize_title(s):
    """Normalizzazione aggressiva applicata sia a palinsesto tivu.tv che ad hist Auditel.
    Port di `get_family_key` (competitor_features.py) + adattamenti per il debug 11_job_forecast.
    """
    if s is None: return ""
    s = str(s).lower().strip()
    s = _strip_accents(s)

    # 1. Parens: qualsiasi contenuto (lettere, anni, "Diretta", marker Auditel)
    s = re.sub(r"\s*\([^)]*\)\s*", " ", s)

    # 2. Prefissi
    for prefix in _TITLE_PREFIXES:
        s = re.sub(prefix, "", s)

    # 3. Pattern episodi numerati "X - X, N"
    s = re.sub(r"\s+-\s+.+,\s*\d+$", "", s)

    # 4. Abbreviazioni puntate (R.I.S. -> ris) - prima della punteggiatura
    s = re.sub(r"\b([a-z])(\.\s*[a-z])+\.?\b",
               lambda m: m.group(0).replace(".", "").replace(" ", ""), s)
    #4.1 Numeri Ordianti con simbolo ° (es. 45°) 
    s = re.sub(r"\b\d+\s*°", " ", s)

    # 4.2 Rimuove "INT." ovunque si trovi nella stringa
    s = re.sub(r"\bint\.\s*", " ", s)

    # 4.3 Intervalli di anni: 2025/20, 2025/2026, 25/26
    s = re.sub(
    r"\b(?:\d{2}|\d{4})\s*/\s*(?:\d{2}|\d{4})\b",
    " ", s)
    # 5. Apostrofi -> rimuovi (camera cafe' -> camera cafe, l'eredita -> leredita)
    s = re.sub(r"['\u2019\u2018`]", "", s)

    # 6. Punteggiatura residua -> spazio
    s = re.sub(r"[^a-z0-9 ]", " ", s)

    # 7. Suffissi editoriali (post-punteggiatura per non confondersi con `,`, `-` ecc.)
    for suffix in _TITLE_SUFFIXES:
        s = re.sub(suffix, "", s)

    # 8. Collapse spazi
    return re.sub(r"\s+", " ", s).strip()


MAPPING_MANUALE = {
    ########################## Mappatura inconsistenze hist Auditel → tivu.tivu
    # ---------------------- Rai 1 ----------------------
    ("Rai 1", "noos lavventura della conoscenza"): "noos",
    ("Rai 1", "premio nastri dargento"): "nastri dargento",
    ("Rai 1", "buongiorno benessere estate il meglio di"): "buongiorno benessere estate il meglio",
    # ---------------------- Rai 2 ----------------------
    ("Rai 2", "tg2 e state con il meglio di costume"): "tg2 estate con costume",
    ("Rai 2", "ncis unita anticrimine"): "ncis",
    ("Rai 2", "tg2 mattina"): "tg2",
    ("Rai 2", "rainews24"): "rai news",
    # ---------------------- Rai 3 ----------------------
    ("Rai 3", "rai parlamento spaziolibero"): "spaziolibero",
    ("Rai 3", "blob di tutto di piu"): "blob",
    ("Rai 3", "sapiens files un solo pianeta"): "sapiens files",
    ("Rai 3", "tg3 mondo"): "tg3 lineanotte",
    ("Rai 3", "elisir estate il meglio di"): "elisir",
    ("Rai 3", "storie incredibili"): "amazing stories",
    ########################## Mappatura inconsistenze tivu.tivu → hist Auditel 
    # ---------------------- Rai 1 ----------------------
    ("Rai 1", "telegiornale"): "tg1",
    ("Rai 1", "che tempo fa"): "meteo",
    ("Rai 1", "tg1 sera"): "tg1",
    ("Rai 1", "tg 1 lis"): "tg1",
    ("Rai 1", "tg1 lis"): "tg1",
    ("Rai 1", "tg 1"): "tg1",
    ("Rai 1", "rainews24"): "rai news",
    ("Rai 1", "techetechete notte"): "techetechete",
    ("Rai 1", "unomattina weekly"): "weekly",
    ("Rai 1", "il meglio di domenica in"): "domenica in il meglio",
    ("Rai 1", "1mattina news"): "uno mattina news",
    ("Rai 1", "unomattina"): "uno mattina",
    ("Rai 1", "doc"): "doc nelle tue mani",
    ("Rai 1", "previsioni sulla viabilita cciss viaggia"): "bollettino viabilita",
    ("Rai 1", "italia a r"): "italia andata e ritorno",
    ("Rai 1", "capri 2"): "capri",
    ("Rai 1", "capri 3"): "capri",
    ("Rai 1", "linea verde meteo verde"): "meteo",
    ("Rai 1", "30 anni porta a porta unestate di ricor"): "porta a porta 30 anni unestate di ricordi",
    ("Rai 1", "codice la vita e digitale") : "codice",
    # ---------------------- Rai 2 ----------------------
    ("Rai 2", "tg2 20 30"): "tg2 sera",
    ("Rai 2", "rainews24"): "rai news",
    ("Rai 2", "tg2 storie i racconti della settimana"): "tg2 storie",
    ("Rai 2", "tg2 eat parade"): "eat parade",
    ("Rai 2", "tg2 lis"): "tg2",
    ("Rai 2", "tg2 week end"): "tg2",
    ("Rai 2", "pizza girls storie di pizze e di donne"): "pizzagirls",
    ("Rai 2", "paradise"): "paradise la finestra sullo showbiz",
    ("Rai 2", "green lovers"): "green lovers 2",
    ("Rai 2", "tg sport giorno"): "tgsport giorno",
    ("Rai 2", "tg sport sera"): "tgsport sera",
    ("Rai 2", "la domenica sportiva mercato"): "la nuova ds",
    ("Rai 2", "crociere di nozze"): "la nave dei sogni",

    #----------------------  Rai 3 ----------------------
    ("Rai 3", "piazza affari"): "tgr piazza affari",
    ("Rai 3", "tg regione meteo"): "tgr meteo",
    ("Rai 3", "rainews24"): "rai news",
    ("Rai 3", "adiretta"): "presa diretta",
    ("Rai 3", "farwest"): "far west",
    ("Rai 3", "tg3 sera"): "tg3",
    ("Rai 3", "tg regione"): "tgr",
    ("Rai 3", "tg3 fuori linea"): "tg3 fuorilinea",
    ("Rai 3", "tg3 linea notte"): "tg3 lineanotte",
    ("Rai 3", "via dei matti numero zero"): "via dei matti n 0",
    ("Rai 3", "ri manda rai tre"): "ri manda raitre",
    # ---------------------- Competitors ----------------------
    ("Canale 5", "meteo"): "il meteo",
    ("Rete 4", "tg4 telegiornale"): "tg4",
    ("Italia 1", "macgyver"): "mac gyver",
    ("Tv8", "tg24 buongiorno"): "sky tg24 buongiorno",
    ("Rete 4", "un esercito di 5 uomini"): "un esercito di cinque uomini"
}

def apply_manual_mapping(canale, prog_norm):
    return MAPPING_MANUALE.get((canale, prog_norm), prog_norm)

def setup_secret(scope: str, key: str, value: str) -> None:
    """Crea uno scope (se nuovo) ed inserisce un secret."""
    w = WorkspaceClient()
    try:
        w.secrets.create_scope(scope=scope)
        print(f"Scope '{scope}' created.")
    except ResourceAlreadyExists:
        print(f"Scope '{scope}' already exists, skipping creation.")
    w.secrets.put_secret(scope=scope, key=key, string_value=value)
    print(f"Secret '{key}' stored in scope '{scope}'.")

def parse_epg(html, canali_target, data_riferimento):
    """Versione FIXATA: scarta la notte del giorno successivo (blocco 2)."""
    soup = BeautifulSoup(html, "html.parser")

    def _to_min(hhmm):
        h, m = hhmm.split(":")
        return int(h) * 60 + int(m)

    rows = []
    for ch_div in soup.find_all("div", class_="q"):
        first_link = ch_div.find("a", attrs={"data-channel": True})
        if not first_link:
            continue
        channel = first_link.get("data-channel", "").strip()
        if channel not in canali_target:
            continue
        programmi = []
        for prog_div in ch_div.find_all("div", class_=re.compile(r"^p")):
            text = prog_div.get_text(" ", strip=True)
            times = re.findall(r"\d{2}:\d{2}", text)
            if not times:
                continue
            titolo = re.sub(r"\d{2}:\d{2}", "", text)
            titolo = re.sub(r"\s+", " ", titolo).strip()
            programmi.append({"titolo": titolo, "orario_inizio": times[0]})

        # ============== TAGLIO GIORNATA TELEVISIVA ==============
        # Primo salto orario all'indietro (es. 23:59 -> 00:55) = inizio giorno d+1.
        taglio = len(programmi)
        for i in range(1, len(programmi)):
            if _to_min(programmi[i]["orario_inizio"]) < _to_min(programmi[i - 1]["orario_inizio"]):
                taglio = i
                break
        programmi_giorno = programmi[:taglio]
        programmi_dopo = programmi[taglio:]   # notte del giorno dopo: scartata
        # =======================================================

        for i in range(len(programmi_giorno)):
            if i < len(programmi_giorno) - 1:
                orario_fine = programmi_giorno[i + 1]["orario_inizio"]
            elif programmi_dopo:
                orario_fine = programmi_dopo[0]["orario_inizio"]
            else:
                orario_fine = "05:59"
            rows.append({
                "Data": data_riferimento,
                "Canale": channel,
                "Programma": programmi_giorno[i]["titolo"],
                "orario_inizio": programmi_giorno[i]["orario_inizio"],
                "orario_fine": orario_fine,
            })
    return rows

In [0]:
def calcola_fascia(hhmm):
    try:
        h = int(hhmm[:2])
        if h < 7:
            return "night"
        if h < 21:
            return "daytime"
        return "primetime"
    except:
        return "night"


def calcola_durata(start, end):
    try:
        s = pd.to_datetime(start, format="%H:%M")
        e = pd.to_datetime(end, format="%H:%M")
        if e < s:
            e += pd.Timedelta(days=1)
        return int((e - s).total_seconds() / 60)
    except:
        return None
    

In [0]:
def fetch_epg(day_offset):
    r = requests.get(
        BASE_URL.format(day_offset=day_offset),
        timeout=15
    )
    r.raise_for_status()
    return r.text

In [0]:
def is_night(val):
    try:
        return int(str(val).split(":")[0]) < 6
    except:
        return False
    

def call_llm(messages, temperature=1):
    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        temperature=temperature    
        )
    return response.choices[0].message.content

def build_prompt(ctx):
    return f"""
Sei un analista TV.

DATI AUDITEL:
{ctx}

Per ogni programma restituisci JSON array:

- gid (obbligatorio)
- evento_forte (true/false)

Regole:
- non inventare dati
- match esatto titoli
"""

def enrich_with_llm(df, context):
    system_prompt = build_prompt(context)
    programs = df.to_dict("records")

    results = {}

    n_batches = (len(programs) + BATCH_SIZE - 1) // BATCH_SIZE

    for b_start in range(0, len(programs), BATCH_SIZE):

        batch = programs[b_start:b_start + BATCH_SIZE]

        lines = []

        for i, p in enumerate(batch):
            gid = b_start + i

            lines.append(
                f"[gid={gid}] Canale: {p['Canale']} | "
                f"Orario: {p['orario_inizio']}-{p['orario_fine']} | "
                f"Programma: {p['Programma']}"
            )

        user_prompt = "Analizza programmi TV:\n\n" + "\n".join(lines)

        print(f"Batch {b_start//BATCH_SIZE + 1}/{n_batches}")

        raw = call_llm([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ])

        try:
            data = json.loads(raw.replace("```json", "").replace("```", ""))
        except Exception as e:
            print("JSON ERROR:", e)
            continue

        for item in data:
            gid = item.get("gid")
            if gid is not None:
                results[int(gid)] = item

    return results

def load_auditel_context(spark_session, canali_filter):
    from pyspark.sql.functions import col, avg

    df = spark_session.table(AUDITEL_TABLE)

    summary = (
        df.filter(col("Canale").isin(canali_filter))
          .groupBy("Programma", "Canale")
          .agg(
              avg("Share").alias("Share_medio"),
              avg("Total_Audience").alias("Audience_medio")
          )
          .orderBy(col("Audience_medio").desc())
          .limit(300)
    )

    return "\n".join(summary.toJSON().collect())